# Synthure — train and deploy on an A100 (end to end, no runtime Hugging Face dependency)

This notebook uploads **Synthure itself** to Colab, trains every Synthure owned model, and
deploys the trained artifacts back to the live app.

**What runs where**

| Stage | Compute | What it does |
|---|---|---|
| Synthetic corpus | CPU | build a large labeled corpus from the real ICD/RxNorm/CMS knowledge, with exact gold labels |
| Tabular models (note type, missing info, readiness, reranker) | CPU | the existing `ml/` harness; fast, no GPU needed |
| **Synthure NER (token classification)** | **A100** | fine tune a transformer on the gold entity spans; this is Synthure's own NER |
| **Synthure ICD coder (optional, bi encoder)** | **A100** | a neural code ranker that upgrades the lexical reranker |
| Export | CPU | quantized ONNX in the transformers.js layout + JSON model params |
| Deploy | CPU | commit the artifacts back to GitHub, which triggers the Vercel deploy |

**About the "no Hugging Face dependency" goal.** Hugging Face is used only to *initialize* the
NER fine tune from a base checkpoint at train time, and that base is configurable (you can point
it at an OpenMed checkpoint, any clinical encoder, or a from scratch config). After export, the
models are plain ONNX files served from your own repository. There is **zero Hugging Face call at
inference or in the deployed app**.

## 0. Confirm the A100

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Set Runtime > Change runtime type > GPU (A100)."

## 1. Upload Synthure to Colab

Clones your repository (it already contains the knowledge artifacts and the `ml/` harness).
Set a token only if the repo is private or if you want to push the trained models back later.

In [ ]:
import os
GITHUB_USER  = "aravinds-kannappan"
GITHUB_REPO  = "Synthure"
GITHUB_TOKEN = ""   # optional: a fine grained PAT with repo write, needed only to push in Step 7

auth = f"{GITHUB_USER}:{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
url  = f"https://{auth}github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
%cd /content
!rm -rf Synthure
!git clone --depth 1 {url}
%cd /content/Synthure
!ls && echo "---" && ls ml && echo "---" && ls frontend/data

## 2. Install dependencies (torch is preinstalled on Colab)

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" \
                "optimum[onnxruntime]>=1.21" onnx onnxruntime "scikit-learn>=1.3" \
                seqeval sentencepiece 2>&1 | tail -4

## 3. Configuration

In [ ]:
from pathlib import Path
ROOT = Path("/content/Synthure")
ML   = ROOT / "ml"
DATA = ROOT / "frontend" / "data"
PUB  = ROOT / "frontend" / "public" / "models"      # ONNX models served to the browser
FEM  = ROOT / "frontend" / "lib" / "models"          # JSON model params for TS inference

# Corpus size: the A100 makes a big corpus cheap. 20k gives the transformer plenty.
CORPUS_N   = 20000

# NER fine tune. Base only initializes training; the exported model is Synthure owned.
# Options: an OpenMed checkpoint, a clinical encoder, or a small general encoder.
NER_BASE   = "OpenMed/OpenMed-NER-DiseaseDetect-TinyMed-65M"
NER_EPOCHS = 3
NER_BATCH  = 32
NER_MAXLEN = 256
NER_OUT    = PUB / "synthure-ner-65m"                # deployed model dir

# Optional neural ICD coder (Step 5). Heavier in the browser; off by default.
TRAIN_ICD_CODER = False
CODER_BASE      = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"

import sys
sys.path.insert(0, str(ML))
print("config ready")

## 4. Build the synthetic corpus and train the tabular models

`generate.py` constructs notes from real conditions, so the gold labels (note type, section
spans, entity spans, ICD/CPT codes, missing fields, readiness) are exact. `train.py` fits the
note type classifier, missing info detector, readiness GBM with isotonic calibration, and the
lexical ICD reranker, then exports them to `frontend/lib/models/`.

In [ ]:
%cd /content/Synthure/ml
!python generate.py {CORPUS_N}
!python train.py
!python evaluate.py
import json
res = json.load(open(ML / "results.json"))
print("\n=== tabular + OpenMed eval ===")
for k, v in res.items():
    print(f"{k:28} {v}")
%cd /content/Synthure

## 5. Train the Synthure NER on the A100 (token classification)

We turn the gold entity spans in the corpus into BIO token labels and fine tune a transformer.
This becomes Synthure's own NER: one model that tags DIAGNOSIS, MEDICATION, SIGN_SYMPTOM,
LAB_VALUE, and PROCEDURE, replacing the two stock OpenMed detectors with a single owned model.

In [ ]:
import json, numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

ENT_TYPES = ["DIAGNOSIS", "MEDICATION", "SIGN_SYMPTOM", "LAB_VALUE", "PROCEDURE"]
LABELS = ["O"] + [f"{p}-{t}" for t in ENT_TYPES for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

def load_jsonl(p): return [json.loads(l) for l in open(p)]
train_rows = load_jsonl(ML / "artifacts/train.jsonl")
val_rows   = load_jsonl(ML / "artifacts/val.jsonl")
test_rows  = load_jsonl(ML / "artifacts/test.jsonl")

tok = AutoTokenizer.from_pretrained(NER_BASE)

def encode(rows):
    feats = []
    for r in rows:
        enc = tok(r["note"], return_offsets_mapping=True, truncation=True, max_length=NER_MAXLEN)
        offs = enc.pop("offset_mapping")
        tags = [label2id["O"]] * len(offs)
        for e in r["entities"]:
            if e["type"] not in ENT_TYPES: continue
            first = True
            for i, (a, b) in enumerate(offs):
                if a == b: continue                      # special token
                if a >= e["start"] and b <= e["end"]:
                    tags[i] = label2id[("B-" if first else "I-") + e["type"]]
                    first = False
        enc["labels"] = tags
        feats.append(enc)
    return Dataset.from_list(feats)

ds_tr, ds_va, ds_te = encode(train_rows), encode(val_rows), encode(test_rows)
print("train tokens example:", len(ds_tr), "val", len(ds_va), "test", len(ds_te))

model = AutoModelForTokenClassification.from_pretrained(
    NER_BASE, num_labels=len(LABELS), id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True)
collator = DataCollatorForTokenClassification(tok)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    true_l, pred_l = [], []
    for pr, la in zip(preds, p.label_ids):
        t, q = [], []
        for pi, li in zip(pr, la):
            if li == -100: continue
            t.append(id2label[int(li)]); q.append(id2label[int(pi)])
        true_l.append(t); pred_l.append(q)
    return {"precision": precision_score(true_l, pred_l),
            "recall": recall_score(true_l, pred_l),
            "f1": f1_score(true_l, pred_l)}

args = TrainingArguments(
    output_dir="/content/ner_ckpt", num_train_epochs=NER_EPOCHS,
    per_device_train_batch_size=NER_BATCH, per_device_eval_batch_size=64,
    learning_rate=3e-5, weight_decay=0.01, warmup_ratio=0.06,
    eval_strategy="epoch", save_strategy="no", logging_steps=50,
    fp16=True, report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                  data_collator=collator, tokenizer=tok, compute_metrics=compute_metrics)
trainer.train()

# Held out test metrics (entity level, seqeval)
pred = trainer.predict(ds_te)
m = compute_metrics(pred)
print("\n=== Synthure NER, held out test ===")
print({k: round(v, 3) for k, v in m.items()})
ner_metrics = {"ner_precision": round(m["precision"], 3),
               "ner_recall": round(m["recall"], 3), "ner_f1": round(m["f1"], 3)}

# Save the fine tuned model + tokenizer to a working dir
FT = Path("/content/synthure_ner_ft"); FT.mkdir(exist_ok=True)
model.save_pretrained(FT); tok.save_pretrained(FT)
print("saved fine tuned NER to", FT)

### 5b. Export the NER to quantized ONNX in the transformers.js layout

Produces `frontend/public/models/synthure-ner-65m/{config.json, tokenizer.json, ...,
onnx/model_quantized.onnx}`, the same layout the app already loads for the OpenMed models.

In [ ]:
import shutil, subprocess
from onnxruntime.quantization import quantize_dynamic, QuantType

work = Path("/content/synthure_ner_onnx")
subprocess.run([sys.executable, "-m", "optimum.exporters.onnx",
                "--model", str(FT), "--task", "token-classification", str(work)], check=True)

NER_OUT.mkdir(parents=True, exist_ok=True)
(NER_OUT / "onnx").mkdir(exist_ok=True)
quantize_dynamic(work / "model.onnx", NER_OUT / "onnx" / "model_quantized.onnx",
                 weight_type=QuantType.QInt8)
for f in work.iterdir():
    if f.suffix == ".json" or f.name in ("vocab.txt", "spm.model", "merges.txt"):
        shutil.copy2(f, NER_OUT / f.name)
size = (NER_OUT / "onnx" / "model_quantized.onnx").stat().st_size / 1e6
print(f"exported {NER_OUT}  ({size:.1f} MB quantized)")
print("labels:", LABELS)

## 6. (Optional) Neural ICD coder on the A100

A bi encoder that embeds ICD 10 CM descriptions once and ranks an entity phrase by cosine
similarity. This upgrades the lexical reranker to semantics. It is **off by default** because a
109M encoder is heavy to run in the browser; enable it only if you plan to serve embeddings
(precompute the code index here, embed the query in the browser). Set `TRAIN_ICD_CODER=True` above.

In [ ]:
if TRAIN_ICD_CODER:
    import gzip, json, numpy as np, torch
    from transformers import AutoTokenizer, AutoModel
    ctok = AutoTokenizer.from_pretrained(CODER_BASE)
    cmodel = AutoModel.from_pretrained(CODER_BASE).cuda().eval()

    with gzip.open(DATA / "icd10cm.json.gz", "rt") as f:
        tab = json.load(f)
    codes = [(c, v[1]) for c, v in tab.items() if v[0] == 1]   # billable only
    def embed(texts, bs=256):
        outs = []
        for i in range(0, len(texts), bs):
            b = ctok(texts[i:i+bs], padding=True, truncation=True, max_length=32, return_tensors="pt").to("cuda")
            with torch.no_grad():
                e = cmodel(**b).last_hidden_state[:, 0]        # CLS
            outs.append(torch.nn.functional.normalize(e, dim=1).cpu().numpy())
        return np.vstack(outs)
    print(f"embedding {len(codes)} billable ICD descriptions on the A100 ...")
    emb = embed([d for _, d in codes]).astype("float16")
    np.savez_compressed(FEM / "icd_index.npz", codes=np.array([c for c, _ in codes]), emb=emb)
    print("wrote", FEM / "icd_index.npz", emb.shape,
          "\nNOTE: browser integration (embed query + cosine) is a separate frontend task.")
else:
    print("ICD coder skipped (TRAIN_ICD_CODER=False).")

## 6b. Trained ICD coder (open data, supersedes the zero-shot embed in 6)

Two trainable stages on the A100, no MIMIC and no credentialing:

1. **retriever** bi-encoder trained on ~269k `phrase -> code` pairs mined from the
   ICD-10-CM index + tabular already in `frontend/data/`.
2. **reranker** cross-encoder fine-tuned on **CodiEsp** (public CC-BY clinical cases),
   then evaluated with the official CodiEsp-D metric (MAP) plus P@k.

See `ml/icd_coder/README.md` for the honest framing. Cell 6 above only embedded
descriptions zero-shot and was never wired in; this replaces it with trained models.

In [ ]:
# Stage 1: train the retriever + build the code index (downloads a biomed backbone)
%cd /content/Synthure/ml/icd_coder
!python train_retriever.py

In [ ]:
# Get CodiEsp (public, CC-BY 4.0). Skip if already unzipped to /content/codiesp.
%cd /content
!wget -q -O codiesp.zip "https://zenodo.org/records/3837305/files/codiesp.zip?download=1" && unzip -q -o codiesp.zip -d /content/codiesp || echo "If the link changed, download from https://zenodo.org/records/3837305 and unzip to /content/codiesp"


In [ ]:
# Stage 2: fine-tune the reranker on CodiEsp and evaluate (MAP, P@k)
%cd /content/Synthure/ml/icd_coder
!python train_reranker.py
import json, pathlib
print(json.loads(pathlib.Path('/content/icd_coder_out/reranker_eval.json').read_text()))

In [ ]:
# Inference demo
%cd /content/Synthure/ml/icd_coder
!python predict.py --note "pt w/ htn and type 2 dm, chronic foot ulcer, sob on exertion" --top 8

## 6c. Faithfulness checker (trained safety net over the Claude writers)

Claude writes the four portals and cannot be fine-tuned on this A100 (closed weights).
This trains the piece that attacks writer hallucination directly: a cross-encoder that
scores each portal sentence against the note + extraction and flags unsupported ones.

Fully open training data: FactCC-style corruptions of Synthure's own synthetic notes
(entity swap, dose change, negation flip, added diagnosis), optionally warm-started on
open FEVER/VitaminC. See `ml/faithfulness/README.md` for the honest framing.

In [ ]:
# Train the faithfulness checker (DeBERTa NLI warm start + synthetic corruptions)
%cd /content/Synthure/ml/faithfulness
!pip -q install "sentencepiece" "datasets>=2.20"
!python train.py --nli
import json, pathlib
print(json.loads(pathlib.Path('/content/faithfulness_out/eval.json').read_text()))

In [ ]:
# Demo: the second claim is an unstated medication and should be flagged
%cd /content/Synthure/ml/faithfulness
!python score.py --note "Patient on lisinopril 10 mg for hypertension." \
    --claim "The patient has hypertension." \
    --claim "The patient takes metformin."

## 7. Deploy: commit the trained models back to GitHub

Pushing to `main` triggers the Vercel deploy. The tabular JSON models are already exported by
Step 4; here we add the ONNX NER and register it. **One reviewed frontend change is required** to
route extraction through the new NER (printed at the end); commit that after you have looked at the
eval numbers above.

In [ ]:
import json
# Write a manifest describing the trained NER so the frontend can pick it up deliberately.
manifest = {
    "id": "synthure-ner-65m",
    "path": "/models/synthure-ner-65m",
    "labels": LABELS,
    "entityTypes": ENT_TYPES,
    "base": NER_BASE,
    "trainedOn": f"{CORPUS_N} synthetic notes",
    "metrics": ner_metrics,
}
(FEM / "ner_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

In [ ]:
# Commit + push (needs GITHUB_TOKEN set in Step 1 with repo write scope).
!git config user.email "colab@synthure.local"
!git config user.name  "Synthure Colab Trainer"
!git add frontend/public/models/synthure-ner-65m frontend/lib/models/*.json ml/results.json ml/artifacts/*.jsonl 2>/dev/null
!git commit -q -m "Train Synthure NER on A100 + refresh tabular models and evals" || echo "nothing to commit"
!git push origin main 2>&1 | tail -3 || echo "push failed — set GITHUB_TOKEN in Step 1"

### 7b. If you prefer not to push from Colab: download the artifacts

In [ ]:
import shutil
shutil.make_archive("/content/synthure_models", "zip", root_dir=str(ROOT / "frontend"),
                    base_dir="public/models/synthure-ner-65m")
# also grab the JSON models
shutil.make_archive("/content/synthure_json_models", "zip", root_dir=str(FEM))
from google.colab import files
files.download("/content/synthure_models.zip")
files.download("/content/synthure_json_models.zip")

## 8. The one frontend wiring change (review, then commit)

The trained NER is deployed as `frontend/public/models/synthure-ner-65m`, but the app still calls
the two OpenMed detectors in `frontend/lib/openmed.ts`. To route extraction through your own model,
register it and use it in `extractEntities`.

**`frontend/lib/openmedModels.ts`** — add:

```ts
synthureNer: {
  local: 'synthure-ner-65m',
  hf: 'synthure (trained on A100)',
  label: 'Synthure NER 65M (int8 ONNX, owned)',
  mb: 66,
},
```

**`frontend/lib/openmed.ts`** — replace the two stock model runs in `extractEntities` with one
call to the unified model. Its labels are `B-/I-DIAGNOSIS | MEDICATION | SIGN_SYMPTOM | LAB_VALUE
| PROCEDURE`, so the BIO aggregator already handles them; map each label suffix straight to the
entity `type`. Keep the OpenMed PII model for de identification.

Commit that change and the deploy will serve a fully Synthure owned extraction stack with no
Hugging Face call at runtime.

### Cost and time on an A100
Corpus generation and the tabular harness take under a minute. The NER fine tune (20k notes, 3
epochs, `NER_MAXLEN=256`) runs in roughly 3 to 6 minutes on an A100. The optional ICD embedding
pass over ~98k descriptions is a couple of minutes. The whole notebook fits comfortably in a
single free A100 session.

## 9. Deploy the trained models to a free Hugging Face Space\n\nThis pushes the app code (`serve/`), the ICD-10-CM tabular, and the trained weights to a Docker Space that serves `/code` and `/faithfulness`. Then set `SYNTHURE_MODEL_API` in Vercel to the Space URL and the app links diagnoses with the trained coder and flags unsupported portal sentences. Unset, the app falls back to its lexical linker unchanged.

In [ ]:
# ── Deploy the trained models to a Hugging Face Space (free CPU) ──────────────
# First create the Space at https://huggingface.co/new-space  (SDK: Docker, CPU Basic,
# Public, name it "synthure-models"). Then set SPACE_ID + paste a write token below.
!pip -q install "huggingface_hub>=0.24"

import os, shutil, pathlib
from huggingface_hub import HfApi, login

SPACE_ID = "legacyaravind/synthure-models"   # <owner>/<space-name>
HF_TOKEN = ""  # https://huggingface.co/settings/tokens (write). Or store as Colab secret HF_TOKEN.
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass
login(token=HF_TOKEN)

REPO = pathlib.Path("/content/Synthure")
stage = pathlib.Path("/content/space_stage")
shutil.rmtree(stage, ignore_errors=True)
stage.mkdir()

# 1) app code from the repo's serve/ folder
for f in ["app.py", "Dockerfile", "requirements.txt", "README.md"]:
    shutil.copy(REPO / "serve" / f, stage / f)
# 2) the FY2026 ICD-10-CM tabular the coder needs for descriptions + billable flags
(stage / "data").mkdir()
shutil.copy(REPO / "frontend" / "data" / "icd10cm.json.gz", stage / "data" / "icd10cm.json.gz")
# 3) trained weights
shutil.copytree("/content/icd_coder_out", stage / "icd_coder_out")
if os.path.isdir("/content/faithfulness_out"):
    shutil.copytree("/content/faithfulness_out", stage / "faithfulness_out")
    print("including the faithfulness checker")
else:
    print("faithfulness_out not found -> shipping the coder only (run cell 6c first to add the checker)")

api = HfApi()
api.create_repo(SPACE_ID, repo_type="space", space_sdk="docker", exist_ok=True)
api.upload_folder(folder_path=str(stage), repo_id=SPACE_ID, repo_type="space",
                  commit_message="Deploy Synthure trained models")

url = "https://" + SPACE_ID.replace("/", "-") + ".hf.space"
print("\nSpace building at: https://huggingface.co/spaces/" + SPACE_ID)
print("Set this in Vercel as  SYNTHURE_MODEL_API =", url)
print("Verify when the build turns green:  curl " + url + "/")
